# YOLOv8n KITTI TensorRT FP32 baseline

Target: `original/yolov8n_kitti/8n_float32_fp32.engine`. Batch 1, 640 × 640. The saved-result table below is historical, not a new benchmark. Fresh benchmark results are saved separately in `tensorrt_baseline_results/8n_float32_fp32`.



- Accuracy: full validation split, batch 1, 640 x 640 square input (`rect=False`). Recall is Ultralytics mean class recall at the validation-selected maximum mean F1 point. Latency uses `conf=0.25`, `iou=0.70`.
- Engine-only latency: CUDA-event duration of TensorRT `execute_async_v3`, with resident GPU input/output buffers and a representative validation image preprocessed before timing. No preprocessing, image upload, NMS, or disk access is counted. One inference is completed before the next begins.
- End-to-end latency: synchronized CPU wall time of Ultralytics `predict` on preloaded OpenCV images, including preprocessing, data transfer, TensorRT execution, NMS and result construction. Disk I/O, initialization and warmup are excluded. The first 100 sorted validation images are cycled for at least 15 seconds and 200 completed images.
- Each phase reports median/P95/mean latency. FPS from inverse mean latency and observed wall throughput are reported separately; engine-only CUDA-event FPS is theoretical service throughput, not application throughput.
- Power/temperature/clocks/utilization and GPU memory are sampled with NVML at approximately 100 ms. Idle power is measured for 5 seconds with the model resident but not executing, after 10 seconds of settling. GPU energy/image is integrated board power divided by completed images in the sustained test window. Incremental energy subtracts the corresponding idle power. These values cover the GPU board, not the full computer, and are window averages rather than energy of an isolated inference.
- NVML total GPU memory includes desktop and other processes. Load delta is approximate; process memory is reported when available (Windows WDDM may return unavailable). Peak memory is the highest sampled value and may miss shorter spikes. PyTorch allocated/reserved memory excludes allocations TensorRT makes outside PyTorch.
- Parameters and FLOPs describe the original PyTorch checkpoint at batch 1 and 640 x 640. THOP uses 2 FLOPs per MAC; these are reference-model operation counts, not TensorRT kernel instruction counts. File sizes use decimal MB.
- Models run sequentially. Background GPU work, thermal state, clock/power management and sampling resolution can affect the measurements. FP32 names identify the exported artifacts; TensorRT may permit TF32 internally.

Missing NVML fields remain `None`, not zero.


In [1]:
from pathlib import Path
import json, importlib.util
ROOT = Path.cwd()
r = json.loads((ROOT/'PTQ/tensorrt_baseline_results/8n_float32_fp32.json').read_text())
e = r['engine_only']['latency']; end = r['end_to_end']; l = end['latency']; t = end['telemetry']
headers = ['Precision','mAP50-95 (%)','mAP50 (%)','Engine median / P95 (ms)','E2E median / P95 (ms)','E2E FPS','Params (M)','GFLOPs','Weights (MB)','Engine (MB)','Approx. GPU load delta (MiB)','E2E GPU power (W)','E2E GPU energy (J/image)']
values = ['FP32',f"{r['accuracy']['map50_95']*100:.2f}",f"{r['accuracy']['map50']*100:.2f}",f"{e['median_ms']:.3f} / {e['p95_ms']:.3f}",f"{l['median_ms']:.3f} / {l['p95_ms']:.3f}",f"{l['fps_observed_wall']:.1f}",f"{r['parameters']/1e6:.3f}",f"{r['gflops']:.2f}",f"{r['weights_size_mb']:.2f}",f"{r['engine_size_mb']:.2f}",f"{end['memory_model_loaded']['gpu_total_used_mib']-end['memory_before_load']['gpu_total_used_mib']:.2f}",f"{t['mean_power_w_time_weighted']:.2f}",f"{t['gpu_energy_j_per_image']:.4f}"]
from IPython.display import display, Markdown
table = '| '+' | '.join(headers)+' |\n| '+' | '.join(['---']*len(headers))+' |\n| '+' | '.join(values)+' |'
display(Markdown('**Historical saved results — not rerun**\n\n'+table))
(ROOT/'8n_float32_fp32_saved_metrics.md').write_text('Historical saved results — not rerun\n\n'+table+'\n',encoding='utf-8')


**Historical saved results — not rerun**

| Precision | mAP50-95 (%) | mAP50 (%) | Engine median / P95 (ms) | E2E median / P95 (ms) | E2E FPS | Params (M) | GFLOPs | Weights (MB) | Engine (MB) | Approx. GPU load delta (MiB) | E2E GPU power (W) | E2E GPU energy (J/image) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| FP32 | 67.55 | 90.49 | 2.623 / 3.151 | 4.264 / 5.625 | 216.8 | 3.012 | 8.20 | 6.24 | 79.82 | 147.19 | 63.91 | 0.2947 |

470

In [2]:
import sys
required = ['torch','tensorrt','ultralytics','cv2','numpy','pynvml']
missing = [name for name in required if importlib.util.find_spec(name) is None]
data_exists = (ROOT/'PTQ/datasets/kitti/images/val').is_dir()
print('Python:', sys.executable)
print('Missing packages:', missing)
print('KITTI validation images available:', data_exists)
print('Fresh benchmark ready:', not missing and data_exists)


Python: C:\Users\dxu1\Desktop\Dissertation\.venv\Scripts\python.exe
Missing packages: []
KITTI validation images available: True
Fresh benchmark ready: True


## Fresh benchmark

Dependencies and the Ultralytics KITTI dataset are installed. This section benchmarks the target FP32 engine on the full validation split, then measures engine latency and E2E latency, memory, power and energy.

Detection Precision and Recall are mean class metrics at the validation-selected maximum mean F1 operating point; the Precision column marked FP32 refers to engine precision.

mAP75 is mean AP at IoU = 0.75 and is distinct from the P75 latency percentile.

In [1]:
from pathlib import Path
import gc, json, os, platform, threading, time
from datetime import datetime, timezone
import cv2
import numpy as np
from IPython.display import display, HTML
import html
import torch
import tensorrt as trt
import pynvml as nv
import ultralytics
from ultralytics import YOLO, settings
import ultralytics.data.utils as data_utils
from ultralytics.utils.torch_utils import get_num_params, get_flops

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'original').is_dir())
OUTPUT = ROOT / 'tensorrt_baseline_results/8n_float32_fp32'
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = 0
IMGSZ = 640
LATENCY_IMAGES = 100
WARMUP = 20
MIN_IMAGES = 200
RUN_SECONDS = 15.0
IDLE_SECONDS = 5.0
IDLE_SETTLE_SECONDS = 10.0
SAMPLE_SECONDS = 0.1
CONF, IOU = 0.25, 0.70
DATASETS = ROOT / 'PTQ/datasets'
settings.update({'datasets_dir': str(DATASETS)})
data_utils.DATASETS_DIR = DATASETS
assert torch.cuda.is_available(), 'CUDA-enabled PyTorch is required'
torch.cuda.set_device(DEVICE)
nv.nvmlInit()
gpu_uuid = torch.cuda.get_device_properties(DEVICE).uuid
GPU = nv.nvmlDeviceGetHandleByUUID(str(gpu_uuid))

def safe(fn, *args):
    try:
        value = fn(*args)
        return value if value < 2**63 else None
    except nv.NVMLError:
        return None

def memory_snapshot():
    info = nv.nvmlDeviceGetMemoryInfo(GPU)
    process = None
    try:
        matches = [p.usedGpuMemory for p in nv.nvmlDeviceGetComputeRunningProcesses(GPU)
                   if p.pid == os.getpid() and p.usedGpuMemory is not None and p.usedGpuMemory < 2**63]
        if matches:
            process = sum(matches) / 2**20
    except nv.NVMLError:
        pass
    return {'gpu_total_used_mib': info.used / 2**20, 'process_used_mib': process,
            'torch_allocated_mib': torch.cuda.memory_allocated(DEVICE) / 2**20,
            'torch_reserved_mib': torch.cuda.memory_reserved(DEVICE) / 2**20}

def sample_gpu():
    try:
        util = nv.nvmlDeviceGetUtilizationRates(GPU)
        gpu_util, memory_util = util.gpu, util.memory
    except nv.NVMLError:
        gpu_util = memory_util = None
    return {'t': time.perf_counter(), 'power_w': (p / 1000 if (p := safe(nv.nvmlDeviceGetPowerUsage, GPU)) is not None else None),
            'temperature_c': safe(nv.nvmlDeviceGetTemperature, GPU, nv.NVML_TEMPERATURE_GPU),
            'sm_clock_mhz': safe(nv.nvmlDeviceGetClockInfo, GPU, nv.NVML_CLOCK_SM),
            'memory_clock_mhz': safe(nv.nvmlDeviceGetClockInfo, GPU, nv.NVML_CLOCK_MEM),
            'gpu_util_percent': gpu_util, 'memory_util_percent': memory_util,
            **memory_snapshot()}

class Sampler:
    def __init__(self):
        self.rows, self.stop_event = [], threading.Event()
    def start(self):
        self.rows.append(sample_gpu())
        self.thread = threading.Thread(target=self.loop, daemon=True)
        self.thread.start()
        return self
    def loop(self):
        while not self.stop_event.wait(SAMPLE_SECONDS):
            self.rows.append(sample_gpu())
    def stop(self):
        self.stop_event.set()
        self.thread.join()
        self.rows.append(sample_gpu())
        return self.rows

def stats(values):
    x = np.asarray([v for v in values if v is not None], dtype=float)
    if not x.size:
        return {'mean': None, 'median': None, 'min': None, 'max': None}
    return {k: float(f(x)) for k, f in [('mean', np.mean), ('median', np.median), ('min', np.min), ('max', np.max)]}

def summarize_telemetry(rows, start, end, images, idle_power):
    power = [r for r in rows if r['power_w'] is not None]
    energy = None
    if len(power) >= 2:
        t = np.array([r['t'] for r in power])
        p = np.array([r['power_w'] for r in power])
        inner = t[(t > start) & (t < end)]
        knots = np.r_[start, inner, end]
        energy = float(np.trapezoid(np.interp(knots, t, p), knots))
    inside = [r for r in rows if start <= r['t'] <= end] or rows
    fields = ['power_w', 'temperature_c', 'sm_clock_mhz', 'memory_clock_mhz',
              'gpu_util_percent', 'memory_util_percent', 'gpu_total_used_mib', 'process_used_mib']
    result = {key: stats([r[key] for r in inside]) for key in fields}
    duration = end - start
    result.update({'sampling_interval_s': SAMPLE_SECONDS, 'samples': len(inside),
                   'duration_s': duration, 'images': images,
                   'mean_power_w_time_weighted': energy / duration if energy is not None else None,
                   'gpu_energy_j': energy,
                   'gpu_energy_j_per_image': energy / images if energy is not None and images else None,
                   'incremental_gpu_energy_j_per_image': (energy - idle_power * duration) / images
                      if energy is not None and idle_power is not None and images else None})
    return result

def idle_measure():
    torch.cuda.synchronize()
    time.sleep(IDLE_SETTLE_SECONDS)  # Allow clocks and NVML utilization counters to settle.
    sampler = Sampler().start()
    start = time.perf_counter()
    time.sleep(IDLE_SECONDS)
    end = time.perf_counter()
    rows = sampler.stop()
    summary = summarize_telemetry(rows, start, end, 0, None)
    return summary['mean_power_w_time_weighted'], summary, rows

def latency_summary(ms, wall_seconds):
    x = np.asarray(ms)
    return {'images': len(ms), 'mean_ms': float(x.mean()), 'median_ms': float(np.median(x)),
            'p75_ms': float(np.percentile(x, 75)), 'p95_ms': float(np.percentile(x, 95)), 'fps_inverse_mean_latency': float(1000 / x.mean()),
            'wall_duration_s': wall_seconds, 'fps_observed_wall': len(ms) / wall_seconds}

def load_engine(path):
    logger = trt.Logger(trt.Logger.ERROR)
    runtime = trt.Runtime(logger)
    with path.open('rb') as f:
        n = int.from_bytes(f.read(4), 'little')
        metadata = {}
        offset = 0
        if 0 < n < path.stat().st_size - 4:
            try:
                metadata = json.loads(f.read(n))
                offset = n + 4
            except (ValueError, UnicodeDecodeError):
                pass
        f.seek(offset)
        engine = runtime.deserialize_cuda_engine(f.read())
    if engine is None:
        raise RuntimeError(f'Failed to deserialize {path}')
    context = engine.create_execution_context()
    buffers = {}
    dtypes = {trt.float32: torch.float32, trt.float16: torch.float16, trt.int32: torch.int32,
              trt.int64: torch.int64, trt.int8: torch.int8, trt.bool: torch.bool}
    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        shape = tuple(engine.get_tensor_shape(name))
        if any(d < 0 for d in shape):
            raise RuntimeError('This baseline requires static batch-1 engines')
        buffers[name] = torch.empty(shape, device=f'cuda:{DEVICE}', dtype=dtypes[engine.get_tensor_dtype(name)])
        context.set_tensor_address(name, buffers[name].data_ptr())
    return runtime, engine, context, buffers, metadata

def clean_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def engine_only(path, image):
    clean_cuda()
    before = memory_snapshot()
    runtime, engine, context, buffers, metadata = load_engine(path)
    from ultralytics.data.augment import LetterBox
    prepared = LetterBox((IMGSZ, IMGSZ), auto=False)(image=image)
    array = np.ascontiguousarray(prepared[:, :, ::-1].transpose(2, 0, 1)[None])
    input_names = [name for name in buffers if engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT]
    if len(input_names) != 1:
        raise RuntimeError('Expected a single image input')
    buffers[input_names[0]].copy_(torch.from_numpy(array).to(DEVICE).to(buffers[input_names[0]].dtype) / 255)
    stream = torch.cuda.current_stream(DEVICE)
    for _ in range(WARMUP):
        if not context.execute_async_v3(stream.cuda_stream):
            raise RuntimeError('TensorRT enqueue failed')
    torch.cuda.synchronize()
    loaded = memory_snapshot()
    idle_power, idle, idle_rows = idle_measure()
    begin, finish = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    timings = []
    sampler = Sampler().start()
    start = time.perf_counter()
    try:
        while len(timings) < MIN_IMAGES or time.perf_counter() - start < RUN_SECONDS:
            begin.record(stream)
            ok = context.execute_async_v3(stream.cuda_stream)
            finish.record(stream)
            finish.synchronize()
            if not ok:
                raise RuntimeError('TensorRT enqueue failed')
            timings.append(begin.elapsed_time(finish))
        end = time.perf_counter()
    finally:
        rows = sampler.stop()
    result = {'latency': latency_summary(timings, end-start), 'memory_before_load': before,
              'memory_model_loaded': loaded, 'memory_after_inference': memory_snapshot(),
              'idle_power_w': idle_power, 'idle_telemetry': idle,
              'telemetry': summarize_telemetry(rows, start, end, len(timings), idle_power),
              'engine_metadata': metadata,
              'io_types': {name: str(engine.get_tensor_dtype(name)) for name in buffers}}
    del context, buffers, engine, runtime
    clean_cuda()
    return result, {'idle': idle_rows, 'running': rows, 'latency_ms': timings}

def end_to_end(path, images):
    clean_cuda()
    before = memory_snapshot()
    model = YOLO(str(path), task='detect')
    kwargs = dict(imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU, rect=False,
                  save=False, verbose=False)
    for i in range(WARMUP):
        model.predict(images[i % len(images)], **kwargs)
    torch.cuda.synchronize()
    loaded = memory_snapshot()
    idle_power, idle, idle_rows = idle_measure()
    timings = []
    sampler = Sampler().start()
    start = time.perf_counter()
    try:
        while len(timings) < MIN_IMAGES or time.perf_counter() - start < RUN_SECONDS:
            torch.cuda.synchronize()
            begin = time.perf_counter()
            prediction = model.predict(images[len(timings) % len(images)], **kwargs)
            torch.cuda.synchronize()
            timings.append((time.perf_counter()-begin)*1000)
            del prediction
        end = time.perf_counter()
    finally:
        rows = sampler.stop()
    result = {'latency': latency_summary(timings, end-start), 'memory_before_load': before,
              'memory_model_loaded': loaded, 'memory_after_inference': memory_snapshot(),
              'idle_power_w': idle_power, 'idle_telemetry': idle,
              'telemetry': summarize_telemetry(rows, start, end, len(timings), idle_power)}
    del model
    clean_cuda()
    return result, {'idle': idle_rows, 'running': rows, 'latency_ms': timings}

def discover():
    engines = sorted(p for p in (ROOT/'original').rglob('*.engine') if p.name.lower().endswith('_fp32.engine'))
    configs = []
    for engine in engines:
        weights = engine.with_name(engine.name[:-len('_fp32.engine')] + '.pt')
        folder = engine.parent.name.lower()
        if 'voc' in folder:
            data, images = 'VOC.yaml', DATASETS/'VOC/images/test2007'
        elif 'kitti' in folder:
            data, images = 'kitti.yaml', DATASETS/'kitti/images/val'
        else:
            raise ValueError(f'Cannot infer dataset for {engine}')
        if not weights.is_file() or not images.is_dir():
            raise FileNotFoundError(f'Missing weights or dataset: {weights}, {images}')
        configs.append({'engine': engine, 'weights': weights, 'data': data, 'images': images})
    if not configs:
        raise FileNotFoundError('No *_fp32.engine models in original')
    return configs

def benchmark_model(config):
    engine, weights = config['engine'], config['weights']
    print(f'\nBENCHMARK: {engine.name}', flush=True)
    paths = sorted(p for p in config['images'].iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'})[:LATENCY_IMAGES]
    images = [cv2.imread(str(p)) for p in paths]
    if not images or any(im is None for im in images):
        raise RuntimeError('Cannot preload validation images')
    reference = YOLO(str(weights), task='detect')
    params = int(get_num_params(reference.model))
    gflops = float(get_flops(reference.model, IMGSZ))
    if not gflops:
        raise RuntimeError('THOP FLOPs profiling failed')
    del reference
    clean_cuda()
    print('Running full validation split...', flush=True)
    detector = YOLO(str(engine), task='detect')
    metrics = detector.val(data=config['data'], split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
                           rect=False, workers=0, plots=False, save_json=False, verbose=False,
                           project=str(OUTPUT/'validation'), name=engine.stem, exist_ok=True)
    accuracy = {'map50_95': float(metrics.box.map), 'map50': float(metrics.box.map50), 'map75': float(metrics.box.map75),
                'precision': float(metrics.box.mp),
                'per_class_precision': np.asarray(metrics.box.p).tolist(),
                'precision_definition': 'Ultralytics mean class precision at its validation-selected max-mean-F1 operating point',
                'recall': float(metrics.box.mr),
                'per_class_recall': np.asarray(metrics.box.r).tolist(),
                'recall_definition': 'Ultralytics mean class recall at its validation-selected max-mean-F1 operating point'}
    del detector, metrics
    clean_cuda()
    print('Measuring engine-only latency and GPU telemetry...', flush=True)
    engine_result, engine_samples = engine_only(engine, images[0])
    print('Measuring end-to-end latency and GPU telemetry...', flush=True)
    e2e_result, e2e_samples = end_to_end(engine, images)
    result = {'model': str(engine.relative_to(ROOT)), 'weights': str(weights.relative_to(ROOT)),
              'dataset': config['data'], 'accuracy': accuracy, 'parameters': params,
              'gflops': gflops, 'gmacs': gflops/2, 'weights_size_mb': weights.stat().st_size/1e6,
              'engine_size_mb': engine.stat().st_size/1e6, 'engine_only': engine_result,
              'end_to_end': e2e_result, 'latency_image_count': len(images)}
    (OUTPUT/f'{engine.stem}_samples.json').write_text(json.dumps({'engine_only':engine_samples,'end_to_end':e2e_samples}, indent=2), encoding='utf-8')
    (OUTPUT/f'{engine.stem}.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps({'model': engine.name, **accuracy,
                      'engine_median_ms':engine_result['latency']['median_ms'],
                      'e2e_median_ms':e2e_result['latency']['median_ms']}, indent=2), flush=True)
    return result

CONFIGS = [{'engine': ROOT/'original/yolov8n_kitti/8n_float32_fp32.engine', 'weights': ROOT/'original/yolov8n_kitti/8n_float32.pt', 'data': 'kitti.yaml', 'images': DATASETS/'kitti/images/val'}]
ENVIRONMENT = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'gpu': torch.cuda.get_device_name(DEVICE),
               'gpu_uuid': str(gpu_uuid), 'python': platform.python_version(), 'torch':torch.__version__,
               'tensorrt':trt.__version__, 'ultralytics':ultralytics.__version__,
               'driver':nv.nvmlSystemGetDriverVersion(), 'imgsz':IMGSZ, 'batch':1,
               'run_seconds_min':RUN_SECONDS, 'idle_seconds':IDLE_SECONDS, 'warmup':WARMUP,
               'idle_settle_seconds':IDLE_SETTLE_SECONDS,
               'sampling_interval_s':SAMPLE_SECONDS, 'pid':os.getpid()}
print(json.dumps(ENVIRONMENT, indent=2))
print('FP32 engines:', [str(c['engine'].relative_to(ROOT)) for c in CONFIGS])


{
  "timestamp_utc": "2026-10-02T09:58:32.096454+00:00",
  "gpu": "NVIDIA RTX A2000",
  "gpu_uuid": "a0aa0f21-a2f1-4018-b27b-236640f36d02",
  "python": "3.14.6",
  "torch": "2.14.0+cu130",
  "tensorrt": "11.3.0.99",
  "ultralytics": "8.4.163",
  "driver": "581.29",
  "imgsz": 640,
  "batch": 1,
  "run_seconds_min": 15.0,
  "idle_seconds": 5.0,
  "warmup": 20,
  "idle_settle_seconds": 10.0,
  "sampling_interval_s": 0.1,
  "pid": 14772
}
FP32 engines: ['original\\yolov8n_kitti\\8n_float32_fp32.engine']


In [2]:
RESULTS = []
for config in CONFIGS:
    result = benchmark_model(config)
    RESULTS.append(result)
    (OUTPUT/'results.json').write_text(json.dumps({'environment': ENVIRONMENT, 'models': RESULTS}, indent=2), encoding='utf-8')
print('Saved:', OUTPUT/'results.json')



BENCHMARK: 8n_float32_fp32.engine


Running full validation split...


Ultralytics 8.4.163  Python-3.14.6 torch-2.14.0+cu130 CUDA:0 (NVIDIA RTX A2000, 6138MiB)


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_fp32.engine for TensorRT inference...


Setting batch=1 input of shape (1, 3, 640, 640)


...

Measuring engine-only latency and GPU telemetry...


Measuring end-to-end latency and GPU telemetry...


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_fp32.engine for TensorRT inference...


{
  "model": "8n_float32_fp32.engine",
  "map50_95": 0.6754988754843907,
  "map50": 0.9048546178977632,
  "map75": 0.7685289783628262,
  "precision": 0.9219737514967707,
  "per_class_precision": [
    0.9518043381340031,
    0.9388699068614117,
    0.9700568129245516,
    0.9284149530657226,
    0.7404650616917753,
    0.9386527107769037,
    0.9436573246776944,
    0.9638689038421043
  ],
  "precision_definition": "Ultralytics mean class precision at its validation-selected max-mean-F1 operating point",
  "recall": 0.8396487021629016,
  "per_class_recall": [
    0.9261721483554933,
    0.9009009009009009,
    0.9461883408071748,
    0.6759299323596833,
    0.6666666666666666,
    0.7824213718030364,
    0.9487179487179487,
    0.8701923076923077
  ],
  "recall_definition": "Ultralytics mean class recall at its validation-selected max-mean-F1 operating point",
  "engine_median_ms": 2.570240020751953,
  "e2e_median_ms": 4.076550001627766
}


Saved: C:\Users\dxu1\Desktop\Dissertation\tensorrt_baseline_results\8n_float32_fp32\results.json


In [3]:
r = RESULTS[0]
e = r['engine_only']['latency']; end = r['end_to_end']; l = end['latency']; t = end['telemetry']
headers = ['Precision','mAP50-95 (%)','mAP50 (%)','mAP75 (%)','Detection Precision (%)','Recall (%)','Engine median / P75 / P95 (ms)','E2E median / P75 / P95 (ms)','E2E FPS','Params (M)','GFLOPs','Weights (MB)','Engine (MB)','Approx. GPU load delta (MiB)','E2E GPU power (W)','E2E GPU energy (J/image)']
values = ['FP32',f"{r['accuracy']['map50_95']*100:.2f}",f"{r['accuracy']['map50']*100:.2f}",f"{r['accuracy']['map75']*100:.2f}",f"{r['accuracy']['precision']*100:.2f}",f"{r['accuracy']['recall']*100:.2f}",f"{e['median_ms']:.3f} / {e['p75_ms']:.3f} / {e['p95_ms']:.3f}",f"{l['median_ms']:.3f} / {l['p75_ms']:.3f} / {l['p95_ms']:.3f}",f"{l['fps_observed_wall']:.1f}",f"{r['parameters']/1e6:.3f}",f"{r['gflops']:.2f}",f"{r['weights_size_mb']:.2f}",f"{r['engine_size_mb']:.2f}",f"{end['memory_model_loaded']['gpu_total_used_mib']-end['memory_before_load']['gpu_total_used_mib']:.2f}",f"{t['mean_power_w_time_weighted']:.2f}",f"{t['gpu_energy_j_per_image']:.4f}"]
from IPython.display import display, Markdown
table = '| '+' | '.join(headers)+' |\n| '+' | '.join(['---']*len(headers))+' |\n| '+' | '.join(values)+' |'
display(Markdown('**Fresh benchmark results**\n\n'+table))

(OUTPUT / 'metrics.md').write_text(table + '\n', encoding='utf-8')


**Fresh benchmark results**

| Precision | mAP50-95 (%) | mAP50 (%) | mAP75 (%) | Detection Precision (%) | Recall (%) | Engine median / P75 / P95 (ms) | E2E median / P75 / P95 (ms) | E2E FPS | Params (M) | GFLOPs | Weights (MB) | Engine (MB) | Approx. GPU load delta (MiB) | E2E GPU power (W) | E2E GPU energy (J/image) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| FP32 | 67.55 | 90.49 | 76.85 | 92.20 | 83.96 | 2.570 / 2.598 / 3.106 | 4.077 / 4.621 / 5.362 | 227.9 | 3.012 | 8.20 | 6.24 | 79.82 | 136.69 | 64.22 | 0.2818 |

553

## YOLOv8n KITTI TensorRT FP16

Same validation and measurement settings as the FP32 baseline. Parameters and GFLOPs refer to the original FP32 checkpoint. Results and raw telemetry are stored separately.

In [2]:
OUTPUT = ROOT / 'tensorrt_baseline_results/8n_float32_fp16'
OUTPUT.mkdir(parents=True, exist_ok=True)
FP16_CONFIG = {
    'engine': ROOT / 'original/yolov8n_kitti/8n_float32_fp16.engine',
    'weights': ROOT / 'original/yolov8n_kitti/8n_float32.pt',
    'data': 'kitti.yaml',
    'images': DATASETS / 'kitti/images/val',
}
FP16_ENVIRONMENT = {**ENVIRONMENT, 'timestamp_utc': datetime.now(timezone.utc).isoformat()}
FP16_RESULT = benchmark_model(FP16_CONFIG)
(OUTPUT / 'results.json').write_text(
    json.dumps({'environment': FP16_ENVIRONMENT, 'models': [FP16_RESULT]}, indent=2), encoding='utf-8')



BENCHMARK: 8n_float32_fp16.engine


Running full validation split...


Ultralytics 8.4.163  Python-3.14.6 torch-2.14.0+cu130 CUDA:0 (NVIDIA RTX A2000, 6138MiB)


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_fp16.engine for TensorRT inference...


Setting batch=1 input of shape (1, 3, 640, 640)


...

Measuring engine-only latency and GPU telemetry...


Measuring end-to-end latency and GPU telemetry...


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_fp16.engine for TensorRT inference...


{
  "model": "8n_float32_fp16.engine",
  "map50_95": 0.6762974507115277,
  "map50": 0.907442910732233,
  "map75": 0.7714129576876614,
  "precision": 0.9116264924789659,
  "per_class_precision": [
    0.9439960923957654,
    0.933607850274517,
    0.9680062135672052,
    0.9123961502156949,
    0.7559888398580789,
    0.9142221719583282,
    0.9227842562132059,
    0.9420103653489316
  ],
  "precision_definition": "Ultralytics mean class precision at its validation-selected max-mean-F1 operating point",
  "recall": 0.8513849841375181,
  "per_class_recall": [
    0.9318515506409139,
    0.9081081081081082,
    0.9497454504327434,
    0.7011202298933026,
    0.6886257461462891,
    0.7982954545454546,
    0.9487179487179487,
    0.8846153846153846
  ],
  "recall_definition": "Ultralytics mean class recall at its validation-selected max-mean-F1 operating point",
  "engine_median_ms": 1.4786560535430908,
  "e2e_median_ms": 2.9848500016669277
}


11957

In [3]:
r = FP16_RESULT
e = r['engine_only']['latency']; end = r['end_to_end']; l = end['latency']; t = end['telemetry']
headers = ['Precision','mAP50-95 (%)','mAP50 (%)','mAP75 (%)','Detection Precision (%)','Recall (%)','Engine median / P75 / P95 (ms)','E2E median / P75 / P95 (ms)','E2E FPS','Params (M)','GFLOPs','Weights (MB)','Engine (MB)','Approx. GPU load delta (MiB)','E2E GPU power (W)','E2E GPU energy (J/image)']
values = ['FP16',f"{r['accuracy']['map50_95']*100:.2f}",f"{r['accuracy']['map50']*100:.2f}",f"{r['accuracy']['map75']*100:.2f}",f"{r['accuracy']['precision']*100:.2f}",f"{r['accuracy']['recall']*100:.2f}",f"{e['median_ms']:.3f} / {e['p75_ms']:.3f} / {e['p95_ms']:.3f}",f"{l['median_ms']:.3f} / {l['p75_ms']:.3f} / {l['p95_ms']:.3f}",f"{l['fps_observed_wall']:.1f}",f"{r['parameters']/1e6:.3f}",f"{r['gflops']:.2f}",f"{r['weights_size_mb']:.2f}",f"{r['engine_size_mb']:.2f}",f"{end['memory_model_loaded']['gpu_total_used_mib']-end['memory_before_load']['gpu_total_used_mib']:.2f}",f"{t['mean_power_w_time_weighted']:.2f}",f"{t['gpu_energy_j_per_image']:.4f}"]
from IPython.display import display, Markdown
table = '| '+' | '.join(headers)+' |\n| '+' | '.join(['---']*len(headers))+' |\n| '+' | '.join(values)+' |'
display(Markdown('**Fresh FP16 benchmark results**\n\n'+table))

(OUTPUT / 'metrics.md').write_text(table + '\n', encoding='utf-8')


**Fresh FP16 benchmark results**

| Precision | mAP50-95 (%) | mAP50 (%) | mAP75 (%) | Detection Precision (%) | Recall (%) | Engine median / P75 / P95 (ms) | E2E median / P75 / P95 (ms) | E2E FPS | Params (M) | GFLOPs | Weights (MB) | Engine (MB) | Approx. GPU load delta (MiB) | E2E GPU power (W) | E2E GPU energy (J/image) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| FP16 | 67.63 | 90.74 | 77.14 | 91.16 | 85.14 | 1.479 / 1.509 / 1.925 | 2.985 / 3.318 / 4.177 | 307.4 | 3.012 | 8.20 | 6.24 | 53.29 | 98.00 | 59.07 | 0.1922 |

552

## YOLOv8n KITTI TensorRT INT8

Same validation and measurement settings as FP32 and FP16. Includes mAP75, detection Precision/Recall and latency P75.

In [2]:
OUTPUT = ROOT / 'tensorrt_baseline_results/8n_float32_int8'
OUTPUT.mkdir(parents=True, exist_ok=True)
INT8_CONFIG = {
    'engine': ROOT / 'original/yolov8n_kitti/8n_float32_int8.engine',
    'weights': ROOT / 'original/yolov8n_kitti/8n_float32.pt',
    'data': 'kitti.yaml',
    'images': DATASETS / 'kitti/images/val',
}
INT8_ENVIRONMENT = {**ENVIRONMENT, 'timestamp_utc': datetime.now(timezone.utc).isoformat()}
INT8_RESULT = benchmark_model(INT8_CONFIG)
(OUTPUT / 'results.json').write_text(
    json.dumps({'environment': INT8_ENVIRONMENT, 'models': [INT8_RESULT]}, indent=2), encoding='utf-8')



BENCHMARK: 8n_float32_int8.engine


Running full validation split...


Ultralytics 8.4.163  Python-3.14.6 torch-2.14.0+cu130 CUDA:0 (NVIDIA RTX A2000, 6138MiB)


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_int8.engine for TensorRT inference...


Setting batch=1 input of shape (1, 3, 640, 640)


...

Measuring engine-only latency and GPU telemetry...


Measuring end-to-end latency and GPU telemetry...


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_int8.engine for TensorRT inference...


{
  "model": "8n_float32_int8.engine",
  "map50_95": 0.6665482289371061,
  "map50": 0.9008407086626855,
  "map75": 0.7572847948965518,
  "precision": 0.9194406180618756,
  "per_class_precision": [
    0.9542304095178918,
    0.9426132138599996,
    0.9793067386196252,
    0.9336550413706238,
    0.7107241000470167,
    0.9292701296938585,
    0.9323758668225268,
    0.9733494445634621
  ],
  "precision_definition": "Ultralytics mean class precision at its validation-selected max-mean-F1 operating point",
  "recall": 0.8319775263344542,
  "per_class_recall": [
    0.9227935072931573,
    0.8878716554392231,
    0.9372197309417041,
    0.6699239956568946,
    0.6388888888888888,
    0.7784090909090909,
    0.9427584635917969,
    0.8779548779548779
  ],
  "recall_definition": "Ultralytics mean class recall at its validation-selected max-mean-F1 operating point",
  "engine_median_ms": 1.5360000133514404,
  "e2e_median_ms": 3.0493000012938865
}


12005

In [3]:
r = INT8_RESULT
e = r['engine_only']['latency']; end = r['end_to_end']; l = end['latency']; t = end['telemetry']
headers = ['Precision','mAP50-95 (%)','mAP50 (%)','mAP75 (%)','Detection Precision (%)','Recall (%)','Engine median / P75 / P95 (ms)','E2E median / P75 / P95 (ms)','E2E FPS','Params (M)','GFLOPs','Weights (MB)','Engine (MB)','Approx. GPU load delta (MiB)','E2E GPU power (W)','E2E GPU energy (J/image)']
values = ['INT8',f"{r['accuracy']['map50_95']*100:.2f}",f"{r['accuracy']['map50']*100:.2f}",f"{r['accuracy']['map75']*100:.2f}",f"{r['accuracy']['precision']*100:.2f}",f"{r['accuracy']['recall']*100:.2f}",f"{e['median_ms']:.3f} / {e['p75_ms']:.3f} / {e['p95_ms']:.3f}",f"{l['median_ms']:.3f} / {l['p75_ms']:.3f} / {l['p95_ms']:.3f}",f"{l['fps_observed_wall']:.1f}",f"{r['parameters']/1e6:.3f}",f"{r['gflops']:.2f}",f"{r['weights_size_mb']:.2f}",f"{r['engine_size_mb']:.2f}",f"{end['memory_model_loaded']['gpu_total_used_mib']-end['memory_before_load']['gpu_total_used_mib']:.2f}",f"{t['mean_power_w_time_weighted']:.2f}",f"{t['gpu_energy_j_per_image']:.4f}"]
from IPython.display import display, Markdown
table = '| '+' | '.join(headers)+' |\n| '+' | '.join(['---']*len(headers))+' |\n| '+' | '.join(values)+' |'
display(Markdown('**Fresh INT8 benchmark results**\n\n'+table))

(OUTPUT / 'metrics.md').write_text(table + '\n', encoding='utf-8')


**Fresh INT8 benchmark results**

| Precision | mAP50-95 (%) | mAP50 (%) | mAP75 (%) | Detection Precision (%) | Recall (%) | Engine median / P75 / P95 (ms) | E2E median / P75 / P95 (ms) | E2E FPS | Params (M) | GFLOPs | Weights (MB) | Engine (MB) | Approx. GPU load delta (MiB) | E2E GPU power (W) | E2E GPU energy (J/image) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| INT8 | 66.65 | 90.08 | 75.73 | 91.94 | 83.20 | 1.536 / 1.580 / 2.294 | 3.049 / 3.662 / 4.337 | 294.4 | 3.012 | 8.20 | 6.24 | 40.78 | 106.00 | 57.75 | 0.1962 |

553